In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 2006
month = 4


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

2006-04-30


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 2006-04-01 12:00:00
end_date 2006-04-02 12:00:00
start_date 2006-04-03 12:00:00
end_date 2006-04-04 12:00:00
start_date 2006-04-05 12:00:00
end_date 2006-04-06 12:00:00
start_date 2006-04-07 12:00:00
end_date 2006-04-08 12:00:00
start_date 2006-04-09 12:00:00
end_date 2006-04-10 12:00:00
start_date 2006-04-11 12:00:00
end_date 2006-04-12 12:00:00
start_date 2006-04-13 12:00:00
end_date 2006-04-14 12:00:00
start_date 2006-04-15 12:00:00
end_date 2006-04-16 12:00:00
start_date 2006-04-17 12:00:00
end_date 2006-04-18 12:00:00
start_date 2006-04-19 12:00:00
end_date 2006-04-20 12:00:00
start_date 2006-04-21 12:00:00
end_date 2006-04-22 12:00:00
start_date 2006-04-23 12:00:00
end_date 2006-04-24 12:00:00
start_date 2006-04-25 12:00:00
end_date 2006-04-26 12:00:00
start_date 2006-04-27 12:00:00
end_date 2006-04-28 12:00:00
start_date 2006-04-29 12:00:00
end_date 2006-04-30 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [00:41<09:43, 41.70s/it]

 13%|███████████▋                                                                            | 2/15 [01:03<06:27, 29.85s/it]

 20%|█████████████████▌                                                                      | 3/15 [01:43<06:52, 34.38s/it]

 27%|███████████████████████▍                                                                | 4/15 [02:15<06:09, 33.56s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [09:40<01:04, 32.47s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [10:01<00:28, 28.95s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [10:23<00:00, 27.07s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [10:23<00:00, 41.59s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_2006-04.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [00:29<06:48, 29.21s/it]

 13%|███████████▋                                                                            | 2/15 [01:00<06:34, 30.37s/it]

 20%|█████████████████▌                                                                      | 3/15 [01:28<05:53, 29.44s/it]

 27%|███████████████████████▍                                                                | 4/15 [01:52<05:00, 27.35s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [06:22<00:58, 29.21s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [06:47<00:27, 27.88s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:09<00:00, 26.11s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:09<00:00, 28.65s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_2006-04.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [00:23<05:27, 23.43s/it]

 13%|███████████▋                                                                            | 2/15 [00:46<05:00, 23.14s/it]

 20%|█████████████████▌                                                                      | 3/15 [01:10<04:43, 23.58s/it]

 27%|███████████████████████▍                                                                | 4/15 [01:37<04:36, 25.10s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [05:37<00:51, 25.63s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [05:59<00:24, 24.60s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:29<00:00, 26.06s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:29<00:00, 25.95s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_2006-04.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [00:35<08:23, 35.94s/it]

 13%|███████████▋                                                                            | 2/15 [01:12<07:53, 36.45s/it]

 20%|█████████████████▌                                                                      | 3/15 [01:34<05:55, 29.58s/it]

 27%|███████████████████████▍                                                                | 4/15 [02:08<05:45, 31.37s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [06:31<00:57, 28.65s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [07:02<00:29, 29.37s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:28<00:00, 28.19s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:28<00:00, 29.88s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_2006-04.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                                                                | 0/15 [00:00<?, ?it/s]

  7%|█████▊                                                                                  | 1/15 [00:28<06:41, 28.66s/it]

 13%|███████████▋                                                                            | 2/15 [01:03<07:00, 32.38s/it]

 20%|█████████████████▌                                                                      | 3/15 [01:26<05:37, 28.14s/it]

 27%|███████████████████████▍                                                                | 4/15 [01:49<04:47, 26.15s/it]

...

 87%|███████████████████████████████████████████████████████████████████████████▍           | 13/15 [06:54<00:59, 29.57s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████▏     | 14/15 [07:18<00:28, 28.05s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:42<00:00, 26.91s/it]

100%|███████████████████████████████████████████████████████████████████████████████████████| 15/15 [07:42<00:00, 30.86s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_2006-04.nc
